# 05 - Model Training

This module trains 6 different models to predict default, and saves each trained model to disk so Module 6 can evaluate them without retraining.

**Why 6 models?** Different techniques have different strengths. We start with the simplest, most explainable one (`LogisticRegression`), then try more flexible ones, to see if the extra complexity is worth it.

**`class_weight="balanced"`:** only 11.6% of borrowers defaulted. Without this setting, a lazy model could just always guess "no default" and look 88% accurate while learning nothing useful. This setting forces the model to pay more attention to the rarer, more important class.

**Input:** `../04-feature-engineering/data/X_train.csv`, `y_train.csv`
**Output:** `models/*.joblib` (one file per trained model)


In [1]:
import os
import numpy as np
import pandas as pd
import joblib

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

DATA_DIR = "../04-feature-engineering/data"
MODEL_DIR = "models"
RANDOM_STATE = 42
os.makedirs(MODEL_DIR, exist_ok=True)

X_train = pd.read_csv(os.path.join(DATA_DIR, "X_train.csv"))
y_train = pd.read_csv(os.path.join(DATA_DIR, "y_train.csv")).iloc[:, 0]

print(f"Loaded training set: {len(X_train):,} rows, {X_train.shape[1]} columns")


Loaded training set: 178,742 rows, 16 columns


In [2]:
numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=[np.number]).columns.tolist()
print(f"Number-columns     : {len(numeric_features)}")
print(f"Text columns left  : {len(categorical_features)} -> {categorical_features}")

numeric_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale",  StandardScaler()),
])
categorical_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", drop="first")),
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features),
])

models = {
    "logistic_regression":    LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE),
    "ridge_logistic_l2":      LogisticRegression(max_iter=2000, penalty="l2", C=1.0, class_weight="balanced", random_state=RANDOM_STATE),
    "lasso_logistic_l1":      LogisticRegression(max_iter=2000, penalty="l1", solver="liblinear", C=1.0, class_weight="balanced", random_state=RANDOM_STATE),
    "decision_tree":          DecisionTreeClassifier(max_depth=6, class_weight="balanced", random_state=RANDOM_STATE),
    "random_forest":          RandomForestClassifier(n_estimators=200, max_depth=12, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
    "gradient_boosting":      GradientBoostingClassifier(random_state=RANDOM_STATE),
}


Number-columns     : 13
Text columns left  : 3 -> ['EmploymentType', 'MaritalStatus', 'LoanPurpose']


In [3]:
for name, estimator in models.items():
    pipe = Pipeline([("prep", preprocessor), ("model", estimator)])
    pipe.fit(X_train, y_train)
    joblib.dump(pipe, os.path.join(MODEL_DIR, f"{name}.joblib"))
    print(f"   trained and saved: {name}")

print(f"\nAll {len(models)} models trained on {len(X_train):,} rows and saved to models/")


   trained and saved: logistic_regression


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


   trained and saved: ridge_logistic_l2


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


   trained and saved: lasso_logistic_l1


   trained and saved: decision_tree


   trained and saved: random_forest


   trained and saved: gradient_boosting

All 6 models trained on 178,742 rows and saved to models/


## How to read this

All 6 models trained without errors and are saved as `.joblib` files in `models/`. Nothing to interpret yet — the real test happens in Module 6, on data none of these models have seen.

## Next module
Module 6 (Model Evaluation) loads these saved models and the held-out test set, and scores each model honestly.
